# Salinas Spatial Benchmark Protocol

**Model-agnostic reproducibility protocol**

This notebook section is shared conceptually by SSFTT, Hybrid, and future comparison models. It is **not SSFTT-specific**.

The fixed benchmark protocol is:

- Salinas dataset
- spatial split only
- 32×32 spatial blocks
- class-aware block assignment
- 60/20/20 target split
- seed 42
- 15×15 centered patches
- original labeled-pixel indices
- no buffered/eroded indices
- training batch size 128
- evaluation batch size 128 for cross-model comparability
- AdamW: lr=1e-3, weight_decay=1e-4
- CrossEntropyLoss
- 50 epochs
- CosineAnnealingLR with T_max=50
- best checkpoint selected by validation Macro-F1

Generated benchmark artifacts use **generic names** so they can be reused by every model.

In [1]:
from pathlib import Path
import sys, os, json, hashlib, platform, subprocess, random, gc

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

SEED = 42
PATCH_SIZE = 15
BLOCK_SIZE = 32
TRAIN_FRACTION = 0.60
VAL_FRACTION = 0.20
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0

OUTPUTS = PROJECT_ROOT / "outputs" / "salinas"
OUTPUTS.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Device: cuda
GPU: NVIDIA GeForce RTX 2050


In [2]:
# Reproducibility controls.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed:", SEED)
print("CuDNN deterministic:", torch.backends.cudnn.deterministic)

Seed: 42
CuDNN deterministic: True


In [3]:
# Repository/environment provenance.
def git(cmd):
    try:
        return subprocess.check_output(
            ["git"] + cmd,
            cwd=PROJECT_ROOT,
            stderr=subprocess.STDOUT,
            text=True,
        ).strip()
    except Exception as e:
        return f"NOT FOUND IN SOURCES: {e}"

print("Git commit:", git(["rev-parse", "HEAD"]))
print("Git branch:", git(["branch", "--show-current"]))
print("Git status:", git(["status", "--short"]))
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

Git commit: a4178bee787808f159f712eb0bdd3d9155ffd29a
Git branch: main
Git status: M notebooks/04_hybrid_model.ipynb
 M notebooks/08_salinas_transformer_training.ipynb
 M notebooks/09_salinas_spatial_split_training.ipynb
 M notebooks/10_salinas_hybrid_ablation.ipynb
 M notebooks/12_salinas_ssftt_spatial.ipynb
 M notebooks/13_salinas_morphformer_spatial.ipynb
 M notebooks/15_salinas_gaht_spatial.ipynb
?? data/Salinas_corrected.mat
?? data/Salinas_gt.mat
?? data/salinas/Salinas_gt.mat
?? notebooks/01_salinas_spatial_benchmark_ssftt.ipynb
?? notebooks/02_salinas_spatial_benchmark_pca_hybrid.ipynb
?? notebooks/19_salinas_pca_ssftt_hybrid_spatial_training_FIXED_v2.ipynb
?? notebooks/data/
?? outputs/ssftt_guided_experiments/
Python: 3.11.5
PyTorch: 2.5.1+cu118
CUDA: 11.8


In [4]:
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"
scene = load_salinas(data_dir, download=True)

rows, cols = scene.labels.shape
labels = scene.labels

print("Cube:", scene.cube.shape)
print("Labels:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(SALINAS_CLASS_NAMES))

Cube: (512, 217, 204)
Labels: (512, 217)
Bands: 204
Classes: 16


In [5]:
def find_spatial_split(
    rows,
    cols,
    labels,
    block_size=32,
    train_fraction=0.60,
    val_fraction=0.20,
    seed=42,
    n_trials=10000,
):
    rng = np.random.default_rng(seed)

    block_rows = rows // block_size
    block_cols = cols // block_size

    block_keys = np.stack(
        [block_rows, block_cols],
        axis=1
    )

    unique_blocks = np.unique(block_keys, axis=0)

    n_blocks = len(unique_blocks)

    n_train = round(train_fraction * n_blocks)
    n_val = round(val_fraction * n_blocks)

    best_score = np.inf
    best_assignment = None

    # Precompute class counts for each block
    block_counts = []

    for br, bc in unique_blocks:

        mask = (
            (block_rows == br) &
            (block_cols == bc)
        )

        counts = np.bincount(
            labels[mask],
            minlength=len(SALINAS_CLASS_NAMES)
        )

        block_counts.append(counts)

    block_counts = np.asarray(block_counts)

    total_class_counts = block_counts.sum(axis=0)

    for _ in range(n_trials):

        permutation = rng.permutation(n_blocks)

        train_ids = permutation[:n_train]
        val_ids = permutation[n_train:n_train + n_val]
        test_ids = permutation[n_train + n_val:]

        train_counts = block_counts[train_ids].sum(axis=0)
        val_counts = block_counts[val_ids].sum(axis=0)
        test_counts = block_counts[test_ids].sum(axis=0)

        # Every class must occur in every split
        if np.any(train_counts == 0):
            continue

        if np.any(val_counts == 0):
            continue

        if np.any(test_counts == 0):
            continue

        # Compare class proportions with overall distribution
        overall_dist = total_class_counts / total_class_counts.sum()

        train_dist = train_counts / train_counts.sum()
        val_dist = val_counts / val_counts.sum()
        test_dist = test_counts / test_counts.sum()

        distribution_error = (
            np.mean(np.abs(train_dist - overall_dist))
            + np.mean(np.abs(val_dist - overall_dist))
            + np.mean(np.abs(test_dist - overall_dist))
        )

        # Compare number of pixels with desired 60/20/20
        total_pixels = len(labels)

        target_train = train_fraction * total_pixels
        target_val = val_fraction * total_pixels
        target_test = (1 - train_fraction - val_fraction) * total_pixels

        size_error = (
            abs(train_counts.sum() - target_train) / total_pixels
            + abs(val_counts.sum() - target_val) / total_pixels
            + abs(test_counts.sum() - target_test) / total_pixels
        )

        score = distribution_error + size_error

        if score < best_score:
            best_score = score

            best_assignment = (
                unique_blocks[train_ids].copy(),
                unique_blocks[val_ids].copy(),
                unique_blocks[test_ids].copy(),
            )

    if best_assignment is None:
        raise RuntimeError(
            "Could not find a valid spatial split. "
            "Try increasing n_trials or changing block_size."
        )

    train_blocks, val_blocks, test_blocks = best_assignment

    def blocks_to_indices(blocks):

        mask = np.zeros(len(rows), dtype=bool)

        for br, bc in blocks:
            mask |= (
                (block_rows == br) &
                (block_cols == bc)
            )

        return np.where(mask)[0]

    train_indices = blocks_to_indices(train_blocks)
    val_indices = blocks_to_indices(val_blocks)
    test_indices = blocks_to_indices(test_blocks)

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    rng.shuffle(test_indices)

    return (
        train_indices,
        val_indices,
        test_indices,
        train_blocks,
        val_blocks,
        test_blocks,
        best_score,
    )

In [6]:
SPLIT_PATH = OUTPUTS / "salinas_spatial_split_seed42.npz"

if SPLIT_PATH.exists():
    split = np.load(SPLIT_PATH)
    train_indices = split["train_indices"]
    val_indices = split["val_indices"]
    test_indices = split["test_indices"]
    print("Loaded existing spatial split:", SPLIT_PATH)
else:
    (
        train_indices,
        val_indices,
        test_indices,
        train_blocks,
        val_blocks,
        test_blocks,
        split_score,
    ) = find_spatial_split(
        rows,
        cols,
        labels,
        block_size=BLOCK_SIZE,
        train_fraction=TRAIN_FRACTION,
        val_fraction=VAL_FRACTION,
        seed=SEED,
        n_trials=10000,
    )
    np.savez(
        SPLIT_PATH,
        train_indices=train_indices,
        val_indices=val_indices,
        test_indices=test_indices,
    )
    print("Created spatial split:", SPLIT_PATH)
    print("Split score:", split_score)

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Loaded existing spatial split: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_spatial_split_seed42.npz
Train: 32337
Validation: 10952
Test: 10840


In [7]:
def sha256_array(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()

split_hashes = {
    "train": sha256_array(train_indices),
    "val": sha256_array(val_indices),
    "test": sha256_array(test_indices),
}

print(json.dumps(split_hashes, indent=2))

with open(OUTPUTS / "salinas_spatial_split_fingerprints.json", "w") as f:
    json.dump(split_hashes, f, indent=2)

{
  "train": "ec3617b38356b1ad4e2b8445b345f87be519a53ca34626484cb93d7b48a171d6",
  "val": "f569fc2e69cbf6bc770e170b9735459d7332d024fe06c433cde990c0f5287d8d",
  "test": "6fb69fcd80676361c70bb751f6f73361dfe8d083b655de33fa7d69424358617e"
}


In [8]:
train_dataset = SalinasPatchDataset(
    scene=scene, indices=train_indices, patch_size=PATCH_SIZE, normalize=True
)
val_dataset = SalinasPatchDataset(
    scene=scene, indices=val_indices, patch_size=PATCH_SIZE, normalize=True
)
test_dataset = SalinasPatchDataset(
    scene=scene, indices=test_indices, patch_size=PATCH_SIZE, normalize=True
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 32337
Validation: 10952
Test: 10840


# Common 30-component PCA preprocessing

PCA is a **generic benchmark preprocessing artifact**, not an SSFTT-specific artifact.

Canonical artifact:
`outputs/salinas/salinas_pca_30.joblib`

It is fitted **only from training patches** and then reused unchanged by every model that uses the PCA-30 input representation.

If the artifact already exists, it is loaded rather than refit.

In [9]:
from sklearn.decomposition import PCA
import joblib

PCA_COMPONENTS = 30
PCA_PATCHES = 2000
PCA_SPECTRAL_VECTORS = 200_000

PCA_PATH = OUTPUTS / "salinas_pca_30.joblib"
PCA_CONFIG_PATH = OUTPUTS / "salinas_pca_30_config.json"

if PCA_PATH.exists() and PCA_CONFIG_PATH.exists():
    pca = joblib.load(PCA_PATH)
    with open(PCA_CONFIG_PATH) as f:
        pca_config = json.load(f)

    if getattr(pca, "n_components_", None) != PCA_COMPONENTS:
        raise ValueError("Existing PCA artifact is not PCA-30.")

    print("Loaded existing common PCA:", PCA_PATH)
else:
    rng = np.random.default_rng(SEED)

    n_selected = min(PCA_PATCHES, len(train_dataset))
    selected_indices = rng.choice(
        len(train_dataset),
        size=n_selected,
        replace=False,
    )

    sampled_patches = np.stack([
        train_dataset[int(i)][0]
        for i in selected_indices
    ])

    sampled_spectra = (
        sampled_patches.transpose(0, 2, 3, 1)
        .reshape(-1, sampled_patches.shape[1])
    )

    if len(sampled_spectra) > PCA_SPECTRAL_VECTORS:
        keep = rng.choice(
            len(sampled_spectra),
            size=PCA_SPECTRAL_VECTORS,
            replace=False,
        )
        sampled_spectra = sampled_spectra[keep]

    pca = PCA(
        n_components=PCA_COMPONENTS,
        svd_solver="randomized",
        random_state=SEED,
    )
    pca.fit(sampled_spectra)

    pca_config = {
        "artifact": "salinas_pca_30.joblib",
        "components": PCA_COMPONENTS,
        "patches_sampled": n_selected,
        "spectral_vectors_used": int(len(sampled_spectra)),
        "svd_solver": "randomized",
        "random_state": SEED,
        "fit_source": "training patches only",
    }

    joblib.dump(pca, PCA_PATH)
    with open(PCA_CONFIG_PATH, "w") as f:
        json.dump(pca_config, f, indent=2)

    print("Created common PCA:", PCA_PATH)

    del sampled_patches, sampled_spectra
    gc.collect()

print("Components:", pca.n_components_)
print("Explained variance:", float(pca.explained_variance_ratio_.sum()))

Created common PCA: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\salinas_pca_30.joblib
Components: 30
Explained variance: 0.9995962977409363


In [10]:
class PCAPatchDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, pca, add_channel_dim=False):
        self.base_dataset = base_dataset
        self.pca = pca
        self.add_channel_dim = add_channel_dim

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        patch, label = self.base_dataset[idx]
        x = patch # [204,H,W]
        C, H, W = x.shape

        z = self.pca.transform(
            x.transpose(1, 2, 0).reshape(-1, C)
        )
        z = z.reshape(H, W, -1).transpose(2, 0, 1).astype(np.float32)

        if self.add_channel_dim:
            z = np.expand_dims(z, axis=0)

        return (
            torch.from_numpy(z),
            torch.tensor(label, dtype=torch.long),
        )

# Official SSFTT spatial experiment

The SSFTT architecture is unchanged from the project/official implementation.

Only the **common benchmark preprocessing** is external to the architecture:
204-band Salinas patches → common PCA-30 representation → SSFTT input shape [B,1,30,H,W].

In [11]:
import subprocess

third_party = PROJECT_ROOT / "third_party" / "SSFTT"
repo_dir = third_party / "HSI_SSFTT"
third_party.mkdir(parents=True, exist_ok=True)

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/zgr6010/HSI_SSFTT.git", str(repo_dir)],
        check=True,
    )

ssftt_code_dir = repo_dir / "cls_SSFTT_IP"
if str(ssftt_code_dir) not in sys.path:
    sys.path.insert(0, str(ssftt_code_dir))

from SSFTTnet import SSFTTnet
print("Official SSFTT source:", ssftt_code_dir)

Official SSFTT source: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\third_party\SSFTT\HSI_SSFTT\cls_SSFTT_IP


In [12]:
pca_train_dataset = PCAPatchDataset(train_dataset, pca, add_channel_dim=True)
pca_val_dataset = PCAPatchDataset(val_dataset, pca, add_channel_dim=True)
pca_test_dataset = PCAPatchDataset(test_dataset, pca, add_channel_dim=True)

train_loader_current = DataLoader(
    pca_train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
val_loader_current = DataLoader(
    pca_val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader_current = DataLoader(
    pca_test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

x0, y0 = next(iter(train_loader_current))
print("SSFTT input batch:", x0.shape)

SSFTT input batch: torch.Size([128, 1, 30, 15, 15])


In [13]:
MODEL_NAME = "ssftt_official_spatial"

model = SSFTTnet(
    in_channels=1,
    num_classes=16,
    num_tokens=4,
    dim=64,
    depth=1,
    heads=8,
    mlp_dim=8,
    dropout=0.1,
    emb_dropout=0.1,
).to(device)

print(model)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

SSFTTnet(
  (conv3d_features): Sequential(
    (0): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1))
    (1): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (conv2d_features): Sequential(
    (0): Conv2d(224, 64, kernel_size=(3, 3), stride=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0): ModuleList(
        (0): Residual(
          (fn): LayerNormalize(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=64, out_features=192, bias=True)
              (nn1): Linear(in_features=64, out_features=64, bias=True)
              (do1): Dropout(p=0.1, inplace=False)
            )
          )
        )
        (1): Residual(
          (fn): LayerNormalize(
            (no

In [14]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)
            total += y.size(0)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(dim=1).cpu().numpy())

    from sklearn.metrics import accuracy_score, f1_score

    return {
        "loss": total_loss / total,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }, np.asarray(y_true), np.asarray(y_pred)

In [15]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device=device, dtype=torch.float32)
        y = y.to(device=device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        total += y.size(0)

    return running_loss / total

In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)

best_macro_f1 = -1.0
best_epoch = 0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        model, train_loader_current, criterion, optimizer
    )

    val_metrics, _, _ = evaluate(
        model, val_loader_current, criterion
    )

    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss {train_loss:.4f} | "
        f"Val Acc {val_metrics['accuracy']:.4f} | "
        f"Val Macro-F1 {val_metrics['macro_f1']:.4f}"
    )

print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_macro_f1)

Epoch 01/50 | Loss 0.1395 | Val Acc 0.8623 | Val Macro-F1 0.9275


Epoch 02/50 | Loss 0.0069 | Val Acc 0.8987 | Val Macro-F1 0.9329


Epoch 03/50 | Loss 0.0032 | Val Acc 0.8904 | Val Macro-F1 0.9314


Epoch 04/50 | Loss 0.0013 | Val Acc 0.9079 | Val Macro-F1 0.9564


Epoch 05/50 | Loss 0.0002 | Val Acc 0.9241 | Val Macro-F1 0.9640


...

Epoch 46/50 | Loss 0.0000 | Val Acc 0.9035 | Val Macro-F1 0.9555


Epoch 47/50 | Loss 0.0000 | Val Acc 0.9050 | Val Macro-F1 0.9559


Epoch 48/50 | Loss 0.0000 | Val Acc 0.9078 | Val Macro-F1 0.9575


Epoch 49/50 | Loss 0.0000 | Val Acc 0.9062 | Val Macro-F1 0.9564


Epoch 50/50 | Loss 0.0000 | Val Acc 0.9055 | Val Macro-F1 0.9561
Best epoch: 10
Best validation Macro-F1: 0.966590060394577


In [17]:
model.load_state_dict(best_state)
model.to(device)

test_metrics, y_test, y_pred = evaluate(
    model, test_loader_current, criterion
)

print("=" * 60)
print("FINAL SPATIAL TEST RESULTS")
print("=" * 60)
print(f"Loss     : {test_metrics['loss']:.6f}")
print(f"Accuracy : {test_metrics['accuracy']:.6f}")
print(f"Macro-F1 : {test_metrics['macro_f1']:.6f}")

FINAL SPATIAL TEST RESULTS
Loss     : 0.146438
Accuracy : 0.972601
Macro-F1 : 0.953433


In [18]:
history_df = pd.DataFrame(history)
history_df.to_csv(
    OUTPUTS / f"{MODEL_NAME}_spatial_history.csv",
    index=False,
)

torch.save(
    {
        "model_state_dict": best_state,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "test_metrics": test_metrics,
        "seed": SEED,
        "split_hashes": split_hashes,
        "patch_size": PATCH_SIZE,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
    },
    OUTPUTS / f"{MODEL_NAME}_spatial_best.pt",
)

np.save(
    OUTPUTS / f"{MODEL_NAME}_spatial_y_test.npy",
    y_test,
)
np.save(
    OUTPUTS / f"{MODEL_NAME}_spatial_y_pred.npy",
    y_pred,
)

run_config = {
    "model": MODEL_NAME,
    "seed": SEED,
    "patch_size": PATCH_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "CosineAnnealingLR(T_max=50)",
    "split_artifact": "salinas_spatial_split_seed42.npz",
    "split_hashes": split_hashes,
    "pca_artifact": "salinas_pca_30.joblib",
}

with open(
    OUTPUTS / f"{MODEL_NAME}_spatial_run_config.json", "w"
) as f:
    json.dump(run_config, f, indent=2)

print("Saved reproducibility artifacts.")

Saved reproducibility artifacts.


In [19]:
# ============================================================
# Efficiency benchmark
# ============================================================

import time
import numpy as np
import pandas as pd
import torch

model.eval()

# ------------------------------------------------------------
# 1. Parameter count
# ------------------------------------------------------------
num_parameters = sum(
    p.numel() for p in model.parameters()
)

parameters_m = num_parameters / 1e6

# Model state size including parameters and buffers
model_size_bytes = sum(
    tensor.numel() * tensor.element_size()
    for tensor in model.state_dict().values()
)
model_size_mb = model_size_bytes / (1024 ** 2)

print(f"Parameters: {num_parameters:,}")
print(f"Parameters (M): {parameters_m:.6f}")
print(f"Model size: {model_size_mb:.4f} MB")


# ------------------------------------------------------------
# 2. Fixed benchmark batch
# ------------------------------------------------------------
benchmark_batch_size = BATCH_SIZE

benchmark_loader = DataLoader(
    pca_test_dataset,
    batch_size=benchmark_batch_size,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

benchmark_x, benchmark_y = next(iter(benchmark_loader))

benchmark_x = benchmark_x.to(
    device=device,
    dtype=torch.float32,
)

print("Benchmark input:", benchmark_x.shape)


# ------------------------------------------------------------
# 3. FLOPs
# ------------------------------------------------------------
flops = np.nan

try:
    from fvcore.nn import FlopCountAnalysis

    model_for_flops = model.to(device)
    model_for_flops.eval()

    with torch.no_grad():
        flop_analysis = FlopCountAnalysis(
            model_for_flops,
            benchmark_x,
        )

    flops = float(flop_analysis.total())

    print(f"FLOPs / batch: {flops:,.0f}")
    print(f"GFLOPs / batch: {flops / 1e9:.6f}")

except ImportError:
    print(
        "fvcore is not installed; FLOPs will be recorded as NaN.\n"
        "Install once with: pip install fvcore"
    )

except Exception as exc:
    print("FLOP calculation failed:", repr(exc))


# ------------------------------------------------------------
# 4. Latency benchmark
# ------------------------------------------------------------
WARMUP_ITERS = 10
BENCHMARK_ITERS = 50

latencies_ms = []

with torch.no_grad():

    # Warm-up
    for _ in range(WARMUP_ITERS):
        _ = model(benchmark_x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Timed runs
    for _ in range(BENCHMARK_ITERS):

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        _ = model(benchmark_x)

        if device.type == "cuda":
            torch.cuda.synchronize()

        elapsed_ms = (time.perf_counter() - start) * 1000.0
        latencies_ms.append(elapsed_ms)


latencies_ms = np.asarray(latencies_ms)

latency_batch_ms = float(np.mean(latencies_ms))
latency_batch_median_ms = float(np.median(latencies_ms))
latency_batch_p95_ms = float(np.percentile(latencies_ms, 95))
latency_batch_std_ms = float(np.std(latencies_ms))

latency_per_sample_ms = (
    latency_batch_median_ms / benchmark_batch_size
)

throughput_samples_sec = (
    benchmark_batch_size / (latency_batch_median_ms / 1000.0)
)


print("\nLatency benchmark")
print("-" * 50)
print(f"Mean batch latency   : {latency_batch_ms:.4f} ms")
print(f"Median batch latency : {latency_batch_median_ms:.4f} ms")
print(f"P95 batch latency    : {latency_batch_p95_ms:.4f} ms")
print(f"Std batch latency    : {latency_batch_std_ms:.4f} ms")
print(f"Per-sample latency   : {latency_per_sample_ms:.6f} ms")
print(f"Throughput           : {throughput_samples_sec:.2f} samples/sec")


# ------------------------------------------------------------
# 5. Peak GPU memory
# ------------------------------------------------------------
peak_gpu_memory_mb = np.nan

if device.type == "cuda":

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

    with torch.no_grad():
        _ = model(benchmark_x)

    torch.cuda.synchronize()

    peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 2)
    )

    print(
        f"Peak GPU memory: "
        f"{peak_gpu_memory_mb:.4f} MB"
    )


# ------------------------------------------------------------
# 6. Combine with classification results
# ------------------------------------------------------------
efficiency_result = {
    "model": "SSFTT (Official)",
    "accuracy": float(test_metrics["accuracy"]),
    "macro_f1": float(test_metrics["macro_f1"]),

    "parameters": int(num_parameters),
    "parameters_m": float(parameters_m),
    "model_size_mb": float(model_size_mb),

    "flops": float(flops),
    "gflops": float(flops / 1e9)
        if not np.isnan(flops) else np.nan,

    "peak_gpu_memory_mb": float(peak_gpu_memory_mb),

    "batch_size": int(benchmark_batch_size),

    "latency_batch_ms": latency_batch_ms,
    "latency_batch_median_ms": latency_batch_median_ms,
    "latency_batch_p95_ms": latency_batch_p95_ms,
    "latency_batch_std_ms": latency_batch_std_ms,

    "latency_per_sample_ms": latency_per_sample_ms,
    "throughput_samples_sec": throughput_samples_sec,
}

efficiency_df = pd.DataFrame([efficiency_result])

display(efficiency_df)

Parameters: 153,224
Parameters (M): 0.153224
Model size: 0.5851 MB
Benchmark input: torch.Size([128, 1, 30, 15, 15])
fvcore is not installed; FLOPs will be recorded as NaN.
Install once with: pip install fvcore

Latency benchmark
--------------------------------------------------
Mean batch latency   : 5.6046 ms
Median batch latency : 5.4664 ms
P95 batch latency    : 6.6143 ms
Std batch latency    : 0.5903 ms
Per-sample latency   : 0.042706 ms
Throughput           : 23415.78 samples/sec
Peak GPU memory: 63.1587 MB


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SSFTT (Official),0.972601,0.953433,153224,0.153224,0.585068,NaN,NaN,63.158691,128,5.604612,5.4664,6.61427,0.590299,0.042706,23415.776356


In [20]:
# Save efficiency benchmark
EFFICIENCY_PATH = OUTPUTS / f"{MODEL_NAME}_efficiency.csv"

efficiency_df.to_csv(
    EFFICIENCY_PATH,
    index=False,
)

print("Saved:", EFFICIENCY_PATH)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\ssftt_official_spatial_efficiency.csv
